In [ ]:
import pandas as pd
import numpy as np
import gc
import os

# Pomáha pri pamäti a odstraňuje SettingWithCopyWarning
pd.options.mode.copy_on_write = True

# Trojfázová sústava - 3 napätia a 3 prúdy
voltage_cols = ['u1_norm', 'u2_norm', 'u3_norm']
current_cols = ['i1_norm', 'i2_norm', 'i3_norm']


In [ ]:
# ============================================================
# KONFIGURÁCIA
# ============================================================

# Cesty k dátam (len nové dáta)
NEW_PARQUET = '../Data/Druhe_data/tuke_poruchy_vzorky.parquet'
NEW_EXCEL = '../Data/Druhe_data/tuke_poruchy_vzorky.csv'
OUTPUT_DIR = '../Data/Export_data/'

# Parametre forecastingu
TRAINING_WINDOW_DAYS = 7
FORECAST_HORIZON_DAYS = 1
MIN_SEGMENT_DAYS = TRAINING_WINDOW_DAYS + FORECAST_HORIZON_DAYS  # = 8

# IQR outlier removal
IQR_K = 3.0

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Min segment: {MIN_SEGMENT_DAYS} dní ({TRAINING_WINDOW_DAYS}d tréning + {FORECAST_HORIZON_DAYS}d forecast)")
print(f"IQR k: {IQR_K}")


---
# ČASŤ 1: NAČÍTANIE DÁT
---

In [ ]:
print("="*60)
print("NAČÍTANIE DÁT (Data/Druhe_data)")
print("="*60)

# 1. Načítanie parquet s meraniami - float32 namiesto float64 šetrí pamäť na polovicu
new_data2 = pd.read_parquet(NEW_PARQUET, engine='pyarrow')
for col in voltage_cols + current_cols:
    if col in new_data2.columns:
        new_data2[col] = new_data2[col].astype('float32')
print(f"Merania: {len(new_data2):,} riadkov, {new_data2['elektromer'].nunique()} elektromerov")

# 2. Načítanie CSV s poruchami
new_data3 = pd.read_csv(NEW_EXCEL, sep=';')
print(f"Poruchy: {len(new_data3)} záznamov, EIC: {new_data3['eic'].nunique()}")

# 3. Zjednotenie typov - oba dataframy musia mať 'elektromer' ako string,
# inak by neskoršie porovnanie nefungovalo
new_data2['elektromer'] = new_data2['elektromer'].astype(int).astype(str).str.strip()
new_data3['elektromer'] = new_data3['elektromer'].astype(int).astype(str).str.strip()

# 4. Namiesto klasického merge robíme priamo označenie cez masku - merge by
# duplikoval riadky pri viacerých poruchách na ten istý elektromer
new_data = new_data2.copy()
del new_data2
gc.collect()

# Začneme tým, že všetko označíme ako Chyba=0, neskôr prepíšeme poruchy na 1
new_data['Chyba'] = np.int8(0)
# t_date = len dátum bez času - potrebujeme to na porovnávanie s rozsahmi porúch
new_data['t_date'] = pd.to_datetime(new_data['t_utc'], utc=True, errors='coerce').dt.tz_convert(None).dt.normalize()

# Pre každý záznam poruchy z CSV označíme zodpovedajúce merania ako Chyba=1
fault_count = 0
for _, row in new_data3.iterrows():
    elm = str(int(row['elektromer']))
    # dayfirst=True - dátumy sú v európskom formáte DD.MM.YYYY
    start = pd.to_datetime(row['porucha_eic_od'], dayfirst=True, errors='coerce')
    end = pd.to_datetime(row['porucha_eic_do'], dayfirst=True, errors='coerce')
    
    # Ak sú oba dátumy chýbajúce, riadok preskočíme
    if pd.isna(start) and pd.isna(end):
        continue
    
    # Maska: vyber len záznamy daného elektromera, ktoré spadajú do intervalu poruchy
    # Ak chýba start alebo end, ten kraj intervalu neaplikujeme (otvorený interval)
    mask = new_data['elektromer'] == elm
    if pd.notna(start):
        mask = mask & (new_data['t_date'] >= start.normalize())
    if pd.notna(end):
        mask = mask & (new_data['t_date'] <= end.normalize())
    
    n = mask.sum()
    fault_count += n
    new_data.loc[mask, 'Chyba'] = np.int8(1)

new_data = new_data.drop(columns=['t_date'])

# Priradenie EIC kódov - vytvoríme slovník elektromer→EIC z CSV a namapujeme ho
elm_to_eic = dict(zip(
    new_data3['elektromer'].astype(str), 
    new_data3['eic'].astype(str)
))
new_data['eic'] = new_data['elektromer'].map(elm_to_eic)
# Ak EIC v slovníku nie je, použijeme aspoň číslo elektromera
new_data['eic'] = new_data['eic'].fillna(new_data['elektromer'])
new_data['eic'] = new_data['eic'].astype(str).str.strip()
new_data['Elektromer'] = pd.to_numeric(new_data['elektromer'], errors='coerce').astype('float32')
new_data = new_data.drop(columns=['elektromer'], errors='ignore')

del new_data3
gc.collect()

n_clean = (new_data['Chyba'] == 0).sum()
n_fault = (new_data['Chyba'] == 1).sum()
print(f"\nChyba=0 (normálne): {n_clean:,}")
print(f"Chyba=1 (poruchy):  {n_fault:,}")
print(f"EIC s poruchami:    {new_data[new_data['Chyba']==1]['eic'].nunique()}")
print(f"Celkovo: {len(new_data):,} riadkov, {new_data['eic'].nunique()} EIC")


In [ ]:
print("\n" + "="*60)
print("PRÍPRAVA DÁT")
print("="*60)

Data = new_data.copy()
del new_data
gc.collect()

# category šetrí pamäť pri stĺpcoch s opakujúcimi sa hodnotami (EIC sa opakuje miliónkrát)
Data['eic'] = Data['eic'].astype('category')
Data['Chyba'] = Data['Chyba'].astype('int8')

# Odstránime duplicity pre kombináciu (eic, t_utc) - jeden elektromer má v jednom čase
# byť len raz; chronologicky zoradíme pre neskoršiu segmentáciu
before = len(Data)
Data = Data.drop_duplicates(subset=['eic', 't_utc'], keep='first')
Data = Data.sort_values(['eic', 't_utc']).reset_index(drop=True)

print(f"Celkovo: {len(Data):,} riadkov, {Data['eic'].nunique()} EIC")
print(f"Duplicít odstránených: {before - len(Data):,}")
print(f"Chyba=0: {(Data['Chyba']==0).sum():,}")
print(f"Chyba=1: {(Data['Chyba']==1).sum():,}")
print(f"Pamäť: {Data.memory_usage(deep=True).sum() / 1e9:.2f} GB")


---
# ČASŤ 2: FILTROVANIE VN ELEKTROMEROV
---

In [ ]:
print("="*60)
print("FILTROVANIE VN ELEKTROMEROV")
print("="*60)

# Pre každý elektromer spočítame medián cez všetky 3 fázy a všetky merania
# (median().median(axis=1) = najprv medián po čase pre každú fázu, potom medián cez fázy)
# Hranica 500V oddelí NN (~230V) od VN (tisíce V)
voltage_median = Data.groupby('Elektromer', observed=True)[voltage_cols].median().median(axis=1)
vn_meters = set(voltage_median[voltage_median > 500].index)
nn_meters = set(voltage_median[voltage_median <= 500].index)

n_before = len(Data)
vn_count = Data['Elektromer'].isin(vn_meters).sum()
nn_count = Data['Elektromer'].isin(nn_meters).sum()

print(f"\nPred filtrom: {n_before:,} záznamov")
print(f"  VN elektromery: {len(vn_meters)} ({vn_count:,} záznamov)")
print(f"  NN elektromery: {len(nn_meters)} ({nn_count:,} záznamov)")

# Necháme len NN - miešať VN s NN do jedného modelu nemá zmysel
Data = Data[~Data['Elektromer'].isin(vn_meters)].copy()
Data = Data.reset_index(drop=True)

print(f"\nPo odstránení VN: {len(Data):,} záznamov, {Data['eic'].nunique()} EIC")
print(f"Odstránených: {n_before - len(Data):,} záznamov ({(n_before - len(Data))/n_before*100:.1f}%)")

del voltage_median, vn_meters, nn_meters
gc.collect()


---
# ČASŤ 3: OPRAVA CHÝBAJÚCICH HODNÔT
---

In [ ]:
print("="*60)
print("OPRAVA CHÝBAJÚCICH HODNÔT")
print("="*60)

print("\nChýbajúce hodnoty pred opravou:")
for col in voltage_cols + current_cols:
    n_nan = Data[col].isna().sum()
    n_zero = (Data[col] == 0).sum() if col in voltage_cols else 0
    print(f"  {col}: {n_nan:,} NaN" + (f", {n_zero:,} nulových" if n_zero > 0 else ""))

# Pre napätie máme štvorstupňový fallback - skúšame ich postupne, kým niečo zaberie:
#   1) lineárna interpolácia v rámci elektromera (max 24 krokov)
#   2) ffill/bfill (kopírovanie predchádzajúcej/nasledujúcej hodnoty)
#   3) medián daného elektromera
#   4) globálny medián (posledná záchrana)
print("\nOprava napätia:")
for col in voltage_cols:
    # Nulové napätie pri NN sieti je nezmysel - prepíšeme na NaN, aby sa interpolovalo
    zero_mask = Data[col] == 0
    Data.loc[zero_mask, col] = np.nan
    
    invalid_count = Data[col].isna().sum()
    if invalid_count == 0:
        print(f"  {col}: OK")
        continue
    
    # Krok 1: interpolácia v rámci jedného elektromera (groupby zabráni "miešaniu" medzi nimi)
    Data[col] = Data.groupby('Elektromer', observed=True)[col].transform(
        lambda x: x.interpolate(method='linear', limit=24, limit_direction='both')
    )
    # Krok 2: zvyšky vyplníme ffill/bfill
    Data[col] = Data.groupby('Elektromer', observed=True)[col].transform(
        lambda x: x.ffill().bfill()
    )
    # Krok 3: čo zostalo, doplníme mediánom daného elektromera
    meter_med = Data.groupby('Elektromer', observed=True)[col].transform('median')
    Data[col] = Data[col].fillna(meter_med)
    # Krok 4: úplne posledná záchrana - globálny medián
    Data[col] = Data[col].fillna(Data[col].median())
    
    still_nan = Data[col].isna().sum()
    print(f"  {col}: opravených {invalid_count:,}, zvyšných NaN: {still_nan:,}")

# Pre prúd - na rozdiel od napätia tu 0 je validná hodnota (žiadny odber),
# takže opravujeme len skutočné NaN. Fallback je 0.
print("\nOprava prúdu:")
for col in current_cols:
    nan_count = Data[col].isna().sum()
    if nan_count == 0:
        print(f"  {col}: OK")
        continue
    
    Data[col] = Data.groupby('Elektromer', observed=True)[col].transform(
        lambda x: x.interpolate(method='linear', limit=24, limit_direction='both')
    )
    Data[col] = Data.groupby('Elektromer', observed=True)[col].transform(
        lambda x: x.ffill().bfill()
    )
    Data[col] = Data[col].fillna(0)
    
    print(f"  {col}: opravených {nan_count:,}")

gc.collect()


---
# ČASŤ 4: ODSTRÁNENIE OUTLIEROV
---

In [ ]:
print("="*60)
print("ODSTRÁNENIE OUTLIEROV (IQR)")
print("="*60)

n_before = len(Data)
# Maska, do ktorej budeme zbierať outliere zo všetkých 6 stĺpcov dohromady
outlier_mask = pd.Series(False, index=Data.index)

print(f"\nIQR metóda s k={IQR_K}:")
print(f"Rozsah: [Q1 - {IQR_K}×IQR, Q3 + {IQR_K}×IQR]")
print()

# Pre každú fázu napätia spočítame Q1, Q3 a IQR (Q3-Q1)
# Hranice nastavíme na Q1 - 3*IQR a Q3 + 3*IQR (klasická IQR metóda s k=3)
print("--- NAPÄTIE ---")
for col in voltage_cols:
    Q1 = Data[col].quantile(0.25)
    Q3 = Data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - IQR_K * IQR
    upper = Q3 + IQR_K * IQR
    
    col_outliers = (Data[col] < lower) | (Data[col] > upper)
    n_out = col_outliers.sum()
    # OR cez všetky stĺpce - stačí, aby bol outlier v jednom z nich a riadok ide preč
    outlier_mask = outlier_mask | col_outliers
    
    print(f"  {col}: Q1={Q1:.1f}, Q3={Q3:.1f}, IQR={IQR:.1f}")
    print(f"         Platný rozsah: [{lower:.1f}, {upper:.1f}]")
    print(f"         Outlierov: {n_out:,} ({n_out/len(Data)*100:.3f}%)")

# To isté pre prúd
print("\n--- PRÚD ---")
for col in current_cols:
    Q1 = Data[col].quantile(0.25)
    Q3 = Data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - IQR_K * IQR
    upper = Q3 + IQR_K * IQR
    
    # Prúd nemôže byť záporný - ak by IQR vypočítal záporný spodný okraj, orežeme ho na 0
    lower = max(lower, 0)
    
    col_outliers = (Data[col] < lower) | (Data[col] > upper)
    n_out = col_outliers.sum()
    outlier_mask = outlier_mask | col_outliers
    
    print(f"  {col}: Q1={Q1:.1f}, Q3={Q3:.1f}, IQR={IQR:.1f}")
    print(f"         Platný rozsah: [{lower:.1f}, {upper:.1f}]")
    print(f"         Outlierov: {n_out:,} ({n_out/len(Data)*100:.3f}%)")

# Outlier riadky odstránime úplne (radšej ako interpolovať - vznikajú tak prirodzené
# medzery, ktoré neskôr pomôžu pri segmentácii)
print(f"\n--- SÚHRN ---")
print(f"Celkovo riadkov s aspoň 1 outlierom: {outlier_mask.sum():,} ({outlier_mask.mean()*100:.2f}%)")

Data = Data[~outlier_mask].copy()
Data = Data.reset_index(drop=True)

print(f"Pred: {n_before:,} → Po: {len(Data):,}")
print(f"Odstránených: {n_before - len(Data):,} riadkov")

# Ešte poistka - ak by aj po IQR ostali záporné prúdy (nemali by), prepíšeme ich na 0
for col in current_cols:
    neg = (Data[col] < 0).sum()
    if neg > 0:
        Data.loc[Data[col] < 0, col] = 0.0
        print(f"\n{col}: {neg:,} záporných → 0")

del outlier_mask
gc.collect()

# Kontrolné štatistiky po čistení
print("\n" + "="*60)
print("ŠTATISTIKY PO ČISTENÍ")
print("="*60)
print(f"\nNN elektromery ({len(Data):,} záznamov):")
for col in voltage_cols:
    print(f"  {col}: min={Data[col].min():.1f}, median={Data[col].median():.1f}, max={Data[col].max():.1f}, std={Data[col].std():.2f}")
for col in current_cols:
    print(f"  {col}: min={Data[col].min():.1f}, median={Data[col].median():.1f}, max={Data[col].max():.1f}, std={Data[col].std():.2f}")


---
# ČASŤ 5: SEGMENTÁCIA
---

In [ ]:
print("="*60)
print("SEGMENTÁCIA NA 10MIN DÁTACH")
print("="*60)

# Segment končí, ak je medzera medzi meraniami väčšia ako 2 vzorky (20 min pri 10min frekvencii)
freq_minutes = 10
step = pd.Timedelta(minutes=freq_minutes)
max_gap = step * 2

Data['t_utc'] = pd.to_datetime(Data['t_utc'], utc=True, errors='coerce')

# Detekcia, kde má začať nový segment - dva dôvody:
#   1) časová medzera väčšia ako max_gap (chýbajú merania)
#   2) zmena stavu Chyba (z 0 na 1 alebo opačne)
g = Data.groupby('eic', observed=True, sort=False)
time_gap = g['t_utc'].diff().gt(max_gap)
chyba_change = g['Chyba'].diff().fillna(0).ne(0)
new_segment = time_gap | chyba_change

# cumsum() spočíta True hodnoty zľava doprava - každý True zvýši segment_id o 1.
# Pre každý EIC sa segment_id ráta zvlášť (ide od 0/1 nanovo).
Data['segment_id'] = new_segment.groupby(Data['eic'], observed=True).cumsum().astype('int32')

del time_gap, chyba_change, new_segment, g
gc.collect()

print(f"Segmentácia dokončená")


In [ ]:
print("\nPočítam štatistiky segmentov...")

# Pre tréning modelov potrebujeme len čisté segmenty
clean_data = Data[Data['Chyba'] == 0]

# Pre každý segment spočítame, kedy začal, skončil a koľko má záznamov
segments = clean_data.groupby(['eic', 'segment_id'], observed=True).agg(
    start_time=('t_utc', 'min'),
    end_time=('t_utc', 'max'),
    num_records=('t_utc', 'count'),
).reset_index()

# Dĺžku segmentu prepočítame z časového rozdielu
segments['duration_hours'] = (segments['end_time'] - segments['start_time']).dt.total_seconds() / 3600
segments['duration_days'] = segments['duration_hours'] / 24
# Koľko 8-dňových okien (7d tréning + 1d forecast) sa do segmentu zmestí - to je počet
# trénovacích príkladov, ktoré z neho vieme vyrobiť. clip(lower=0) ošetrí krátke segmenty.
segments['num_training_examples'] = (segments['duration_days'] - MIN_SEGMENT_DAYS + 1).clip(lower=0).astype(int)

del clean_data
gc.collect()

print(f"Celkovo segmentov (Chyba=0): {len(segments):,}")


In [ ]:
# Necháme len segmenty dlhé aspoň 8 dní - kratšie nestačia ani na jeden trénovací príklad
valid_segments = segments[segments['duration_days'] >= MIN_SEGMENT_DAYS].copy()

print(f"\n" + "="*60)
print(f"POUŽITEĽNÉ SEGMENTY (≥ {MIN_SEGMENT_DAYS} dní)")
print("="*60)
print(f"Segmentov: {len(valid_segments):,}")
print(f"EIC: {valid_segments['eic'].nunique()}")
print(f"Celkovo trénovacích príkladov: {valid_segments['num_training_examples'].sum():,}")

print(f"\nŠtatistiky dĺžky segmentov (dni):")
print(valid_segments['duration_days'].describe().round(1))


In [ ]:
# Označíme priamo v dátach, ktoré riadky patria do použiteľných segmentov.
# Set zip(...) je rýchly lookup - O(1) namiesto O(n).
valid_keys = set(zip(valid_segments['eic'], valid_segments['segment_id']))
Data['valid_segment'] = [1 if (e, s) in valid_keys else 0 for e, s in zip(Data['eic'], Data['segment_id'])]
Data['valid_segment'] = Data['valid_segment'].astype('int8')

print(f"\nZáznamy v použiteľných segmentoch: {Data['valid_segment'].sum():,} ({Data['valid_segment'].mean()*100:.1f}%)")


---
# ČASŤ 6: AGREGÁCIA
---

In [ ]:
def aggregate_data(df, freq, name):
    print(f"  Agregácia {name}...")
    
    # Pre napätie a prúd použijeme medián (odolnejší voči zvyškom outlierov ako priemer).
    # Pre Chyba a valid_segment použijeme max - ak v okne bola aspoň jedna 1, výsledok je 1.
    agg_dict = {col: 'median' for col in voltage_cols + current_cols}
    agg_dict['Chyba'] = 'max'
    agg_dict['valid_segment'] = 'max'
    agg_dict['Elektromer'] = 'first'
    
    # pd.Grouper s freq=freq vytvorí časové okná (napr. 30min alebo 1h).
    # Zoskupujeme podľa eic + segment_id + okno, aby sa nemiešali rôzne segmenty.
    result = (
        df.groupby(['eic', 'segment_id', pd.Grouper(key='t_utc', freq=freq)], observed=True)
        .agg(agg_dict)
        .reset_index()
    )
    
    print(f"    → {len(result):,} riadkov")
    return result


In [ ]:
print("="*60)
print("AGREGÁCIA")
print("="*60)

# Exportujeme aj poruchové dáta (Chyba=1) - tie sa použijú na test anomaly detection.
# Pre tréning sa použijú len dáta z platných segmentov (valid_segment=1).
export_mask = (Data['valid_segment'] == 1) | (Data['Chyba'] == 1)
Data_export = Data[export_mask].copy()

n_clean = (Data_export['Chyba'] == 0).sum()
n_fault = (Data_export['Chyba'] == 1).sum()
print(f"\nDáta na export: {len(Data_export):,} riadkov")
print(f"  Chyba=0 (tréning/validácia): {n_clean:,}")
print(f"  Chyba=1 (testovanie anomálií): {n_fault:,}")

# Tri granularity: 10min (originál), 30min, 1h
data_10min = Data_export.copy()
data_30min = aggregate_data(Data_export, '30min', '30min')
data_1h = aggregate_data(Data_export, '1h', '1h')

del Data_export
gc.collect()


In [ ]:
def recalc_segments(df, freq_name):
    # Po agregácii má každý segment menej riadkov, preto si prepočítame štatistiky
    seg = df.groupby(['eic', 'segment_id'], observed=True).agg(
        start_time=('t_utc', 'min'),
        end_time=('t_utc', 'max'),
        num_records=('t_utc', 'count'),
    ).reset_index()
    seg['duration_hours'] = (seg['end_time'] - seg['start_time']).dt.total_seconds() / 3600
    seg['duration_days'] = seg['duration_hours'] / 24
    print(f"  {freq_name}: {len(seg):,} segmentov, priemer {seg['duration_days'].mean():.1f} dní")
    return seg

print("\nPrepočet segmentov pre agregované dáta:")
segments_30min = recalc_segments(data_30min, '30min')
segments_1h = recalc_segments(data_1h, '1h')


---
# ČASŤ 7: EXPORT
---

In [ ]:
print("="*60)
print("EXPORT DO: " + OUTPUT_DIR)
print("="*60)

# Tri parquet súbory pre tri granularity + jeden CSV s metadátami segmentov
print("\nExportujem dáta...")
data_10min.to_parquet(f'{OUTPUT_DIR}/forecast_10min.parquet', engine='pyarrow', index=False)
data_30min.to_parquet(f'{OUTPUT_DIR}/forecast_30min.parquet', engine='pyarrow', index=False)
data_1h.to_parquet(f'{OUTPUT_DIR}/forecast_1h.parquet', engine='pyarrow', index=False)

print("Exportujem segmenty...")
valid_segments.to_csv(f'{OUTPUT_DIR}/segments.csv', sep=';', index=False)

# Vypíšeme veľkosti vygenerovaných súborov pre kontrolu
print(f"\nExportované:")
for f in ['forecast_10min.parquet', 'forecast_30min.parquet', 'forecast_1h.parquet', 'segments.csv']:
    path = f'{OUTPUT_DIR}/{f}'
    if os.path.exists(path):
        size = os.path.getsize(path) / 1e6
        print(f"  {f}: {size:.1f} MB")


In [ ]:
print("\n" + "="*70)
print("ZÁVEREČNÉ ZHRNUTIE")
print("="*70)

print(f"""
KONFIGURÁCIA:
  Trénovacie okno: {TRAINING_WINDOW_DAYS} dní
  Predikčný horizont: {FORECAST_HORIZON_DAYS} deň
  Min segment: {MIN_SEGMENT_DAYS} dní
  IQR k: {IQR_K}
  Typ siete: len NN (VN odstránené)

DÁTA (len nové dáta - tuke_poruchy_vzorky, Chyba=0 + Chyba=1):
  10min: {len(data_10min):,} riadkov
  30min: {len(data_30min):,} riadkov
  1h:    {len(data_1h):,} riadkov

SEGMENTY:
  Použiteľných: {len(valid_segments):,}
  EIC: {valid_segments["eic"].nunique()}
  
VÝSTUP: {OUTPUT_DIR}/
""")
